# Huấn luyện & Đánh giá PPO cho Super Mario Bros (`mario_ppo`) — Colab / Kaggle

Notebook tự động hóa cho Nhóm 1 (Việt, Sơn, Quang, Tín):
- Tự nhận diện môi trường **Google Colab** (mount Google Drive) hoặc **Kaggle** (`/kaggle/working`) để không mất checkpoint khi ngắt phiên 12 giờ.
- Cài đặt gói `mario_ppo` và kiểm tra `pytest -q` trước khi chạy.
- Tự động tìm checkpoint `step_*.pt` mới nhất để nối tiếp bằng `resume=...` (ưu tiên hơn `init_ckpt`).
- Đánh giá chuẩn theo giao thức chấm (`--episodes 20 --seed 2026`) và vẽ biểu đồ báo cáo 300 DPI.

Các tệp cấu hình sẵn trong `configs/`: `1-1.yaml`, `1-4.yaml`, `4-1.yaml`, `7-1.yaml`, `ablation_gamma099.yaml`, `ablation_notricks.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = Path("/kaggle/working").exists()

if IS_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive/mario_ppo_workspace")
elif IS_KAGGLE:
    WORK_ROOT = Path("/kaggle/working/mario_ppo_workspace")
else:
    WORK_ROOT = Path.cwd()

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Platform: {'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'} | WORK_ROOT={WORK_ROOT}")

In [ ]:
import subprocess

def get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    if IS_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name) or ""
        except Exception:
            return ""
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name) or ""
        except Exception:
            return ""
    return ""

GITHUB_TOKEN = get_secret("GITHUB_TOKEN")
WANDB_API_KEY = get_secret("WANDB_API_KEY")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

REPO_URL = (
    f"https://{GITHUB_TOKEN}@github.com/TTins1702/DRL-Mo-Supermario-group-1.git"
    if GITHUB_TOKEN
    else "https://github.com/TTins1702/DRL-Mo-Supermario-group-1.git"
)
BRANCH = "main"

if IS_COLAB:
    REPO_DIR = Path("/content/DRL-Mo-Supermario-group-1")
elif IS_KAGGLE:
    REPO_DIR = Path("/kaggle/working/DRL-Mo-Supermario-group-1")
elif (Path.cwd() / "mario_ppo").is_dir():
    REPO_DIR = Path.cwd()
else:
    REPO_DIR = WORK_ROOT / "DRL-Mo-Supermario-group-1"

def run_cmd(cmd, cwd=None):
    display_cmd = [
        s.replace(GITHUB_TOKEN, "***") if GITHUB_TOKEN else s
        for s in map(str, cmd)
    ]
    print(">>>", " ".join(display_cmd), flush=True)
    with subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
        rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, display_cmd)

if not (REPO_DIR / ".git").exists():
    run_cmd(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)])
elif REPO_DIR != Path.cwd():
    run_cmd(["git", "fetch", "origin"], cwd=REPO_DIR)
    run_cmd(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run_cmd(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)

os.chdir(REPO_DIR)
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "wandb"])
run_cmd([sys.executable, "-c", "import os, torch; print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'cpus:', os.cpu_count())"])

In [ ]:
run_cmd([sys.executable, "-m", "pytest", "-q"])

In [ ]:
CONFIG = "configs/1-1.yaml"  # Đổi thành 1-4.yaml, 4-1.yaml, 7-1.yaml, ablation_gamma099.yaml, ablation_notricks.yaml
SEED = 1
RUN_NAME = f"{Path(CONFIG).stem}_seed{SEED}"
RUN_DIR = WORK_ROOT / "runs" / RUN_NAME
INIT_CKPT = None  # Ví dụ: str(WORK_ROOT / "runs/1-1_seed1/checkpoints/step_004000000.pt") khi transfer sang 4-1/7-1
USE_WANDB = True
WANDB_PROJECT = "mario-ppo"
VIDEO_EVERY = 500_000  # mỗi 500k bước (~25-40 phút) đánh giá & xuất 1 video lên W&B; có thể giảm xuống 250_000
EXTRA_OVERRIDES = []  # Ví dụ: ["algo.total_steps=6000000"]

ckpts = sorted((RUN_DIR / "checkpoints").glob("step_*.pt")) if (RUN_DIR / "checkpoints").exists() else []
latest_ckpt = ckpts[-1] if ckpts else None

train_cmd = [
    sys.executable, "-m", "mario_ppo.train",
    "--config", CONFIG,
    f"seed={SEED}",
    f"run_dir={RUN_DIR}",
    f"checkpoint_every={VIDEO_EVERY}",
]
if USE_WANDB:
    if not os.environ.get("WANDB_API_KEY"):
        import wandb
        wandb.login()
    train_cmd.extend([
        "wandb.enabled=true",
        f"wandb.project={WANDB_PROJECT}",
        f"wandb.name={RUN_NAME}",
    ])
if latest_ckpt is not None:
    print(f"Phát hiện checkpoint cũ -> tự động resume từ {latest_ckpt}")
    train_cmd.append(f"resume={latest_ckpt}")
elif INIT_CKPT:
    print(f"Học chuyển giao (global_step=0) từ init_ckpt={INIT_CKPT}")
    train_cmd.append(f"init_ckpt={INIT_CKPT}")
train_cmd.extend(EXTRA_OVERRIDES)
print("Lệnh huấn luyện:", " ".join(train_cmd))

In [ ]:
run_cmd(train_cmd, cwd=REPO_DIR)

In [ ]:
import json
from omegaconf import OmegaConf

cfg_path = RUN_DIR / "config.yaml" if (RUN_DIR / "config.yaml").exists() else Path(CONFIG)
cfg_used = OmegaConf.load(cfg_path)
level = str(cfg_used.level)
final_ckpts = sorted((RUN_DIR / "checkpoints").glob("step_*.pt"))
assert final_ckpts, f"Không tìm thấy checkpoint trong {RUN_DIR / 'checkpoints'}"
eval_ckpt = final_ckpts[-1]

EVAL_JSON = RUN_DIR / f"ket-qua-{level}.json"
EVAL_VIDEO = RUN_DIR / f"{level}.mp4"

run_cmd([
    sys.executable, "-m", "mario_ppo.evaluate",
    "--ckpt", str(eval_ckpt),
    "--level", level,
    "--episodes", "20",
    "--seed", "2026",
    "--out", str(EVAL_JSON),
    "--video", str(EVAL_VIDEO),
], cwd=REPO_DIR)

summary = json.loads(EVAL_JSON.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2, ensure_ascii=False))

if USE_WANDB:
    import wandb
    wb_eval = wandb.init(project=WANDB_PROJECT, name=f"{RUN_NAME}_official_eval", config=summary)
    eval_payload = {
        "official_eval/flag_rate": float(summary["flag_rate"]),
        "official_eval/mean_x_pos": float(summary["mean_x_pos"]),
    }
    if EVAL_VIDEO.exists():
        eval_payload["official_eval/video"] = wandb.Video(str(EVAL_VIDEO), caption=f"{level} official eval", format="mp4")
    wb_eval.log(eval_payload)
    wb_eval.finish()

In [ ]:
from IPython.display import Image, display

CURVES_PNG = RUN_DIR / f"curves_{level}.png"
FAILURES_PNG = RUN_DIR / f"failures_{level}.png"

run_cmd([
    sys.executable, "-m", "scripts.plot_report", "curves",
    "--csv", str(RUN_DIR / "progress.csv"),
    "--out", str(CURVES_PNG),
], cwd=REPO_DIR)
run_cmd([
    sys.executable, "-m", "scripts.plot_report", "failures",
    "--json", str(EVAL_JSON),
    "--level", level,
    "--out", str(FAILURES_PNG),
], cwd=REPO_DIR)

display(Image(filename=str(CURVES_PNG)))
display(Image(filename=str(FAILURES_PNG)))